# SHIVA-1 — multi-domain research dashboard

Interactive tour of the full meta-science loop:

**alternate physics → universe simulation → emergence detection → AI technology evolution → real-world mapping → CAD.**

Every cell runs on a laptop CPU. `QUICK = True` keeps the whole notebook to a few minutes; set it to `False`
for research-grade budgets. Figures are written to `shiva_dashboard/output/` and displayed inline.
The end-to-end, reproducible version of everything below is `python -m shiva_core run --profile standard`.

In [1]:
import sys, json, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "shiva_dashboard" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
OUT = ROOT / "shiva_dashboard" / "output"
OUT.mkdir(parents=True, exist_ok=True)

import numpy as np
from IPython.display import Image, Markdown, Math, display

from shiva_core import visualizer as viz
from shiva_core.physics_mutator import PhysicsMutator, SymbolicPhysics, noether_analysis
from shiva_core.universe_simulator import SimulationConfig, run_universe
from shiva_core.emergence_detector import EmergenceDetector
from shiva_core.ai_brain import ShivaBrain
from shiva_core.tech_evolver import DOMAINS, PhysicsContext, TechEvolver, GenerativeBracketDesigner
from shiva_core.real_world_mapper import RealWorldMapper
from shiva_core.cad_generator import CADGenerator
from shiva_core.pipeline import showcase_genomes

QUICK = True
SEED = 11
B = dict(n=64 if QUICK else 160, steps=1200 if QUICK else 4000, tech_gens=10 if QUICK else 30,
         tech_pop=20 if QUICK else 36, search_gens=1 if QUICK else 5, search_pop=4 if QUICK else 10)

def show(path, width=900):
    display(Image(filename=str(path), width=width))

## 1 · Physics mutation engine
Five universes along the gravitational escalation ladder. Each law is a SymPy potential; forces are its exact
negative gradient, so energy conservation is never assumed by hand.

In [2]:
genomes = showcase_genomes(SEED)
for g in genomes:
    sym = SymbolicPhysics(g)
    display(Markdown(f"**{g.name}**  —  effective Gauss-law dimension $d_{{eff}} = {sym.effective_dimension():.2f}$"))
    display(Math(r"U(r) = " + sym.latex()["U_gravity"]))
viz.plot_force_laws(genomes, OUT / "force_laws.png", labels=[g.name for g in genomes])
show(OUT / "force_laws.png")

**newtonian (inverse-square)**  —  effective Gauss-law dimension $d_{eff} = 3.00$

<IPython.core.display.Math object>

**inverse-k (k = 2.5, d_eff = 3.5)**  —  effective Gauss-law dimension $d_{eff} = 3.50$

<IPython.core.display.Math object>

**tensor (anisotropic metric Q)**  —  effective Gauss-law dimension $d_{eff} = 3.00$

<IPython.core.display.Math object>

**fractal (log-periodic, eps = 0.4)**  —  effective Gauss-law dimension $d_{eff} = 3.00$

<IPython.core.display.Math object>

**yukawa (lambda = 1)**  —  effective Gauss-law dimension $d_{eff} = 3.00$

<IPython.core.display.Math object>

### Symmetries → conservation laws (Noether)
Conservation laws are *derived*, never mutated directly: breaking a symmetry is the only way a conserved quantity
disappears.

In [3]:
rows = ["| universe | energy | momentum | angular momentum | time reversal |", "|---|---|---|---|---|"]
for g in genomes:
    n = noether_analysis(g)
    rows.append(f"| {g.name} | {n.energy} | {n.momentum} ({n.momentum_kind}) | {len(n.angular_axes)} axes | {n.time_reversal} |")
display(Markdown("\n".join(rows)))
orb = SymbolicPhysics(genomes[3]).orbital_analysis()
display(Markdown(f"Fractal gravity: **{orb['n_stable_bands']} separate bands** of stable circular orbits between r = 0.05 and 20."))

| universe | energy | momentum | angular momentum | time reversal |
|---|---|---|---|---|
| newtonian (inverse-square) | True | True (mechanical) | 3 axes | True |
| inverse-k (k = 2.5, d_eff = 3.5) | True | True (mechanical) | 3 axes | True |
| tensor (anisotropic metric Q) | True | True (mechanical) | 0 axes | True |
| fractal (log-periodic, eps = 0.4) | True | True (mechanical) | 3 axes | True |
| yukawa (lambda = 1) | True | True (mechanical) | 3 axes | True |

Fractal gravity: **17 separate bands** of stable circular orbits between r = 0.05 and 20.

## 2 · Universe simulation (particles + rigid bodies + continuum fields)

In [4]:
g = genomes[3]                      # fractal gravity
cfg = SimulationConfig(n_particles=B["n"], steps=B["steps"], initial_condition="cold_collapse", seed=SEED)
t0 = time.time()
run = run_universe(g, cfg, fields=True, rigid=True, field_resolution=48 if QUICK else 64)
print(f"simulated in {time.time()-t0:.1f}s;  Lyapunov exponent = {run.particles.lyapunov:.2f}")
print(json.dumps({k: {kk: v[kk] for kk in ("predicted", "relative_drift", "consistent")}
                  for k, v in run.evaluation.conservation.items()}, indent=1))
viz.plot_particles(run.particles, OUT / "particles.png", title=g.name); show(OUT / "particles.png")
viz.plot_conservation(run.particles, OUT / "conservation.png"); show(OUT / "conservation.png", 650)

simulated in 6.3s;  Lyapunov exponent = 11.34
{
 "energy": {
  "predicted": true,
  "relative_drift": 0.0002180412928519783,
  "consistent": true
 },
 "momentum": {
  "predicted": true,
  "relative_drift": 7.554705198459413e-16,
  "consistent": true
 },
 "angular_momentum": {
  "predicted": true,
  "relative_drift": 5.601945769001429e-16,
  "consistent": true
 }
}


In [5]:
viz.plot_fields(run.fields, OUT / "fields.png"); show(OUT / "fields.png")
viz.plot_rigid(run.rigid, OUT / "rigid.png"); show(OUT / "rigid.png")

## 3 · Emergence detector

In [6]:
rep = EmergenceDetector(seed=SEED).analyze(run)
print(rep.summary())
viz.plot_emergence(rep, OUT / "emergence.png"); show(OUT / "emergence.png", 650)
viz.plot_correlation(rep, OUT / "correlation.png"); show(OUT / "correlation.png", 500)

emergent richness R = 0.525  (stability 1.00)
  self_organization               0.37 #######              DETECTED
  oscillation                     0.02                      
  attractor                       0.27 #####                DETECTED
  chaos                           1.00 #################### DETECTED
  fractal                         0.47 #########            DETECTED
  symmetry_breaking               0.31 ######               DETECTED
  clustering                      0.00                      
  pattern_formation               1.00 #################### DETECTED
  coherent_vortices               1.00 #################### DETECTED
  rotational_symmetry_breaking    0.50 ##########           DETECTED


## 4 · AI brain: universe search
Evolution over physics genomes with a novelty archive, UCB1-chosen mutation operators and 1/5th-rule step-size
adaptation. (Use the pipeline for held-out validation of the winner.)

In [7]:
brain = ShivaBrain(seed=SEED, verbose=False,
                   sim_config=SimulationConfig(n_particles=40, steps=600, record_every=15, seed=SEED))
search = brain.search(generations=B["search_gens"], population=B["search_pop"], seed_genomes=genomes[:B["search_pop"]])
best = search["best"]
print(best.genome.describe()); print(best.brief())
viz.plot_search(search["history"], OUT / "search.png"); show(OUT / "search.png", 500)

PhysicsGenome fractal (log-periodic, eps = 0.4) (gen 0, dim 3)
  gravity : fractal, G=1, k=2, Q-eig=[1.0, 1.0, 1.0], eps=0.4, gamma=0.5, omega=6, beta=2, octaves=3
  em      : off (alpha/alpha0=1)
  inertia : newtonian, masses=equal
  thermo  : gamma=0, T=0, noise=gaussian(q=1), active=0@v0=0.5, adiabatic=1.4
  fluid   : nu=0.001, h=1, lambda=1, N-eig=[1.0, 1.0], drag=0|v|^2
  interact: instantaneous, species=1
  cosmos  : Lambda=0, H=0
{'genome_id': 'be9a0838ba70', 'law': 'fractal', 'fitness': 0.4183, 'richness': 0.4385, 'novelty': 0.702, 'stability': 1.0, 'consistency': 1.0, 'feasibility': 0.816}


## 5 · Technology evolution under the winner's physics

In [8]:
alien = PhysicsContext.from_genome(best.genome)
print(alien.description)
tech = {}
for k, name in enumerate(DOMAINS):
    tech[name] = TechEvolver(name, alien, seed=SEED + k).evolve(generations=B["tech_gens"], pop_size=B["tech_pop"])
    c = tech[name].champion
    print(f"{name:18s} feasible={c.feasible} " + " ".join(f"{o}={v:.2f}" for o, v in c.objectives.items()))
viz.plot_pareto(tech, OUT / "pareto.png"); show(OUT / "pareto.png")
viz.plot_design_gallery({n: r.champion.params for n, r in tech.items()}, OUT / "gallery.png"); show(OUT / "gallery.png")

genome be9a0838ba70: g=1.7 g0, alpha/alpha0=1, drag n=2, gamma=1.4


propulsion_nozzle  feasible=True efficiency=0.75 stability=0.40 adaptability=0.74 mappability=0.87 cosmic=0.76


structure_truss    feasible=True efficiency=0.41 stability=0.98 adaptability=0.41 mappability=0.71 cosmic=0.41
energy_flywheel    feasible=True efficiency=0.48 stability=1.00 adaptability=0.49 mappability=0.75 cosmic=0.51


drone_frame        feasible=True efficiency=0.45 stability=0.72 adaptability=0.37 mappability=1.00 cosmic=0.59
pump_impeller      feasible=True efficiency=0.71 stability=0.80 adaptability=0.71 mappability=0.86 cosmic=0.30


gear_pair          feasible=True efficiency=0.76 stability=1.00 adaptability=0.76 mappability=0.88 cosmic=0.76


cosmic_habitat     feasible=True efficiency=0.33 stability=1.00 adaptability=0.00 mappability=0.00 cosmic=0.00
cosmic_lightsail   feasible=True efficiency=1.00 stability=0.83 adaptability=1.00 mappability=1.00 cosmic=0.87


## 6 · Real-world mapping, build instructions and CAD

In [9]:
cad = CADGenerator(str(OUT / "cad"), render_openscad=False)
mapper = RealWorldMapper(n_monte_carlo=60, seed=SEED, cad_generator=cad)
protos = {n: mapper.map(r.champion, out_dir=str(OUT / "cad"), name=n) for n, r in tech.items()}
for n, p in protos.items():
    print(f"{n:18s} {p.material_name:40s} feasible={p.feasible} fidelity={p.fidelity:.3f} reliability={p.reliability:.0%} -> {Path(p.cad['scad']).name}")
display(Markdown(protos["gear_pair"].report()))

propulsion_nozzle  Stainless steel 304                      feasible=True fidelity=0.966 reliability=73% -> propulsion_nozzle.scad
structure_truss    CFRP quasi-isotropic laminate            feasible=True fidelity=0.997 reliability=100% -> structure_truss.scad
energy_flywheel    Aluminium 7075-T6                        feasible=True fidelity=0.949 reliability=55% -> energy_flywheel.scad
drone_frame        CFRP quasi-isotropic laminate            feasible=True fidelity=0.949 reliability=100% -> drone_frame.scad
pump_impeller      Aluminium 6061-T6                        feasible=True fidelity=0.997 reliability=100% -> pump_impeller.scad
gear_pair          Alloy steel 4140 (Q&T)                   feasible=True fidelity=0.926 reliability=100% -> gear_pair.scad
cosmic_habitat     CFRP unidirectional (fibre direction)    feasible=True fidelity=0.973 reliability=55% -> cosmic_habitat.scad
cosmic_lightsail   Polyimide film (Kapton HN)               feasible=True fidelity=0.992 reliability=100

### gear_pair prototype (Alloy steel 4140 (Q&T))
- feasible on Earth: **True**  (Monte-Carlo reliability 100%)
- fidelity to alien design: 0.926; efficiency alien 0.759 -> real 0.764
- mass 7.87 kg, est. material+parts cost $1,014

| parameter | alien | real |
|---|---|---|
| module | 0.002039 | 0.002 |
| z1 | 38 | 38 |
| face_factor | 12.65 | 13 |
| pressure_angle | 23.31 | 25 |
| shift | -0.08542 | -0.1 |

| predicted metric | mean | std |
|---|---|---|
| mesh_efficiency | 0.9963 | 1.9e-05 |
| contact_ratio | 1.56 | 0 |
| bending_stress_MPa | 75.13 | 0.69 |
| contact_stress_MPa | 458.6 | 9.7 |
| power_density_W_kg | 634.6 | 10 |
| center_distance_mm | 152 | 0 |
| z2 | 114 | 0 |
| ratio | 3 | 0 |
| pitch_velocity_m_s | 5.969 | 8.9e-16 |
| face_width_mm | 26.03 | 0.24 |
| mass_kg | 7.881 | 0.13 |

Build instructions:
1. Turn blanks in Alloy steel 4140 (Q&T); pinion z = 38, gear z = 114, module 2.00 mm, pressure angle 25.0 deg, profile shift x1 = -0.10 / x2 = +0.10.
2. Hob teeth (or wire-EDM/print for prototypes); through-harden or nitride as specified for steels.
3. Grind/finish flanks to ISO 1328 grade 6-7; deburr tips.
4. Mount on parallel shafts at centre distance 152.00 mm (no shift sum), check backlash and contact pattern with marking compound.
5. Run-in under 25 % load with ISO VG 220 oil, then load-test to 1.25x rated torque.
6. Inspect all critical dimensions against the CAD model (tolerances as noted) before assembly.
7. Proof-test to 1.2x the design load/pressure/speed behind a barrier before use; record results.
8. Planning values only: confirm material certificates and have a qualified engineer review load-bearing, rotating or pressurised parts.

## 7 · Generative design (CPPN-NEAT + lattice FEM)

In [10]:
br = GenerativeBracketDesigner(ctx=alien, seed=SEED).run(generations=10 if QUICK else 30, pop_size=24 if QUICK else 40)
print(f"stiffness relative to solid plate: {br['fitness']:.3f} at volume fraction {br['metrics']['volume_fraction']:.2f}")
viz.plot_bracket(br, OUT / "bracket.png"); show(OUT / "bracket.png")

stiffness relative to solid plate: 0.061 at volume fraction 0.45


## 8 · Dashboard

In [11]:
viz.plot_dashboard(run, rep, search, tech, protos, OUT / "dashboard.png"); show(OUT / "dashboard.png", 1000)

## 9 · Reproducible pipeline results
If you have run `python -m shiva_core run --profile standard --seed 42`, its auto-generated results are shown here.

In [12]:
res = ROOT / "shiva_output" / "standard_seed42" / "results.md"
alt = ROOT / "shiva_docs" / "results" / "results.md"
display(Markdown(res.read_text() if res.exists() else alt.read_text() if alt.exists() else "_no pipeline run found_"))

# SHIVA-1 results (seed 42, profile standard, v0.1.0)

## Noether validation
| universe | energy (pred / drift) | momentum | angular momentum | consistency |
|---|---|---|---|---|
| newtonian | conserved / 2.2e-06 | conserved / 4.3e-16 | conserved / 4.5e-16 | 100% |
| inverse-k 2.5 | conserved / 3.6e-05 | conserved / 3.1e-16 | conserved / 7.4e-16 | 100% |
| anisotropic gravity | conserved / 3.6e-06 | conserved / 4.3e-16 | broken / 7.1e-02 | 100% |
| axisymmetric gravity | conserved / 2.0e-06 | conserved / 2.2e-16 | conserved / 3.0e-16 | 100% |
| relativistic inertia c=1.5 | conserved / 1.3e-06 | conserved / 3.7e-16 | conserved / 7.4e-16 | 100% |
| anisotropic inertia | conserved / 2.5e-06 | conserved / 7.7e-16 | broken / 2.9e-02 | 100% |
| time-varying G | broken / 1.8e-01 | conserved / 3.3e-16 | conserved / 4.6e-16 | 100% |
| Lambda < 0 (AdS box) | conserved / 2.2e-06 | broken / 3.4e-16 | conserved / 2.2e-16 | 100% |
| Coulomb + B field | conserved / 1.4e-06 | conserved / 8.6e-08 | conserved / 5.0e-08 | 100% |
| non-reciprocal species | broken / 8.7e-03 | broken / 9.2e-02 | broken / 1.8e-02 | 100% |
| retarded gravity c=3 | broken / 3.1e-02 | broken / 4.0e-02 | broken / 1.9e-02 | 100% |
| Langevin bath | broken / 1.2e+00 | broken / 2.1e-01 | broken / 1.4e-01 | 100% |

## Gravity ladder (symbolic analysis)
| law | d_eff | stable circular orbit (r=1) | apsidal angle | stability bands | field eqs verified |
|---|---|---|---|---|---|
| newtonian (inverse-square) | 3.00 | True | 3.1416 | 1 | True |
| inverse-k (k = 2.5, d_eff = 3.5) | 3.50 | True | 4.4429 | 1 | True |
| tensor (anisotropic metric Q) | 3.00 | True | 3.1416 | 1 | True |
| fractal (log-periodic, eps = 0.4) | 3.00 | True | 0.4046 | 17 | True |
| yukawa (lambda = 1) | 3.00 | True | 4.4429 | 1 | True |

## Universe search
Evaluations: 59; best fitness 0.5764 (after CMA-ES: 0.5885).

```
PhysicsGenome shiva-best-02d1d5e62596 (gen 4, dim 3)
  gravity : tensor, G=1, k=2.5, Q-eig=[0.319, 1.489, 2.106]
  em      : off (alpha/alpha0=1)
  inertia : newtonian, masses=equal
  thermo  : gamma=0, T=0, noise=gaussian(q=1), active=0@v0=0.5, adiabatic=1.4
  fluid   : nu=0.001, h=1, lambda=1, N-eig=[1.0, 1.0], drag=0|v|^2
  interact: instantaneous, species=1, nonreciprocity=0.000, sigma=0.3, core=0.347@0.0503
  cosmos  : Lambda=0.178, H=0
```

| genome | gravity law | fitness | richness | novelty | feasibility |
|---|---|---|---|---|---|
| 02d1d5e62596 | tensor | 0.5764 | 0.603 | 0.0 | 0.828 |
| fafd2198ebbb | inverse_k | 0.5687 | 0.6046 | 0.2433 | 0.761 |
| e6f3c9c13912 | inverse_k | 0.5687 | 0.6046 | 0.0 | 0.761 |
| 01ff22092e7e | inverse_k | 0.5635 | 0.5963 | 0.0 | 0.78 |
| 777685b15bf3 | inverse_k | 0.5613 | 0.5963 | 0.1925 | 0.766 |
| a16e7aeca948 | tensor | 0.5575 | 0.579 | 0.2248 | 0.857 |
| 348a0d1f7bfd | tensor | 0.5326 | 0.5454 | 0.2522 | 0.9 |
| 39d470500810 | inverse_k | 0.5323 | 0.553 | 0.16 | 0.85 |
| 9def1ddd84b3 | inverse_k | 0.5298 | 0.5539 | 0.1702 | 0.826 |
| 12c149ba130c | inverse_k | 0.5227 | 0.542 | 0.132 | 0.857 |

Held-out validation at final resolution (fresh seeds):

| candidate | gravity law | search fitness | validated richness |
|---|---|---|---|
| e27822b81220 | tensor | 0.589 | 0.501 +/- 0.073 |
| 02d1d5e62596 | tensor | 0.576 | 0.654 +/- 0.019 |
| fafd2198ebbb | inverse_k | 0.569 | 0.621 +/- 0.087 |
| e6f3c9c13912 | inverse_k | 0.569 | 0.621 +/- 0.087 |

## Emergence: best universe vs Newtonian baseline
| phenomenon | best | baseline |
|---|---|---|
| self organization | 1.000 | 1.000 |
| oscillation | 0.344 | 0.000 |
| attractor | 0.034 | 0.127 |
| chaos | 0.993 | 0.866 |
| fractal | 0.957 | 0.872 |
| symmetry breaking | 0.500 | 0.674 |
| clustering | 0.199 | 0.122 |
| pattern formation | 0.997 | 0.997 |
| coherent vortices | 0.722 | 0.722 |
| rotational symmetry breaking | 0.500 | 0.500 |
| **richness R (seed 0)** | **0.667** | **0.612** |
| richness mean +/- sd, 5 held-out test seeds | 0.624 +/- 0.026 | 0.558 +/- 0.083 |

Welch t-test on held-out test seeds (best vs baseline richness): t = 1.69, p = 0.154.

- best: Lyapunov 4.935, D2 = 0.60 (Poisson ref 2.72), FoF groups 2, clustered fraction 0.32, xi slope 2.6371261484307857, dominant period 0.21729729729727942
- baseline: Lyapunov 2.008, D2 = 1.32 (Poisson ref 2.43), FoF groups 1, clustered fraction 0.24, xi slope 2.699021229793555, dominant period 0.535999999999956

## Technology champions (alien physics)
| domain | feasible | eff | stab | adapt | map | cosmic | pareto | PPCA joined front |
|---|---|---|---|---|---|---|---|---|
| propulsion_nozzle | True | 0.77 | 0.47 | 0.76 | 0.88 | 0.78 | 49 | 13 |
| structure_truss | True | 0.60 | 1.00 | 0.59 | 0.80 | 0.60 | 46 | 17 |
| energy_flywheel | True | 0.48 | 1.00 | 0.47 | 0.74 | 0.49 | 55 | 19 |
| drone_frame | True | 1.00 | 0.99 | 0.93 | 1.00 | 0.60 | 51 | 15 |
| pump_impeller | True | 0.82 | 1.00 | 0.82 | 0.91 | 0.35 | 43 | 7 |
| gear_pair | True | 0.76 | 0.99 | 0.75 | 0.88 | 0.76 | 47 | 11 |
| cosmic_habitat | True | 0.65 | 0.87 | 0.64 | 0.82 | 0.26 | 44 | 8 |
| cosmic_lightsail | True | 1.00 | 1.00 | 1.00 | 1.00 | 0.90 | 40 | 7 |

Adaptive-design agent (mean improvement per 8-edit episode across 5 physics contexts):

| domain | trained policy | random edits | untrained |
|---|---|---|---|
| propulsion_nozzle | 0.3341 | 0.0508 | 0.0000 |
| structure_truss | 0.1980 | -0.0417 | 0.0000 |
| energy_flywheel | 0.1671 | -0.0232 | 0.0000 |

Generative bracket: fitness 0.157 under alien physics, 0.157 for the same shape on Earth; volume fraction 0.45.

## Real-world prototypes
| domain | material | feasible | fidelity | efficiency alien -> real | MC reliability | mass kg | cost |
|---|---|---|---|---|---|---|---|
| propulsion_nozzle | Stainless steel 304 | True | 0.959 | 0.77 -> 0.76 | 62% | 1.2 | $1,820 |
| structure_truss | CFRP quasi-isotropic laminate | True | 0.994 | 0.60 -> 0.67 | 100% | 2.96 | $248 |
| energy_flywheel | Aluminium 7075-T6 | True | 0.949 | 0.48 -> 0.56 | 69% | 1.43e+03 | $57,140 |
| drone_frame | Aluminium 6061-T6 | True | 0.961 | 1.00 -> 1.00 | 100% | 3.05 | $704 |
| pump_impeller | Aluminium 6061-T6 | True | 0.968 | 0.82 -> 0.82 | 100% | 1.33 | $1,936 |
| gear_pair | Alloy steel 4140 (Q&T) | True | 0.871 | 0.76 -> 0.74 | 100% | 9.39 | $1,019 |
| cosmic_habitat | CFRP unidirectional (fibre direction) | True | 0.999 | 0.65 -> 0.64 | 63% | 9.49e+06 | $474,465,952 |
| cosmic_lightsail | Polyimide film (Kapton HN) | True | 0.973 | 1.00 -> 1.00 | 100% | 332 | $761,711 |

Total wall time: 662 s.
